# 🔥 WiDS 2026 — Wildfire Survival Analysis: Optuna-Tuned Ensemble Guide

> **Public Leaderboard Score: 96.38%** ✅

---

## What This Notebook Covers

This notebook is a **complete, end-to-end guide** to the WiDS 2026 wildfire survival analysis competition. We walk through:

1. **Exploratory Data Analysis** — understanding 316 wildfire events with survival data
2. **Feature Engineering** — 44 carefully crafted features from raw wildfire dynamics
3. **Adversarial Validation** — diagnosing train/test distribution shift
4. **Feature Ablation Study** — rigorous feature selection to avoid overfitting (221 samples!)
5. **Optuna Hyperparameter Tuning** — for 3 survival models (CompGBM, GBSA, RSF)
6. **Diversity-Aware Ensembling** — a critical lesson about _why_ tuning all models can hurt

---

## Competition Overview

Given the **first 5 hours** of wildfire spread data, predict four probabilities:

| Target | Meaning |
|--------|----------|
| `prob_12h` | Fire comes within 5km of evacuation zone by 12h |
| `prob_24h` | ... by 24h |
| `prob_48h` | ... by 48h |
| `prob_72h` | ... by 72h |

This is **right-censored survival data** — if a fire doesn't reach an evacuation zone within 72h, we only know it *hadn't hit yet*, not that it never will.

### Competition Metric
$$\text{Score} = 0.3 \times C\text{-index} + 0.7 \times (1 - \text{WBS})$$

where $\text{WBS} = 0.3 \cdot B_{24h} + 0.4 \cdot B_{48h} + 0.3 \cdot B_{72h}$ (weighted Brier score).

**Key constraint**: probabilities must be monotone — `prob_12h ≤ prob_24h ≤ prob_48h ≤ prob_72h`

---

### Dataset Size Warning ⚠️
- **Train: 221 rows** (69 hits, 152 censored)
- **Test: 95 rows**

With only 221 samples, **overfitting is the #1 enemy**. Every design decision in this notebook is shaped by this constraint.

## 1. Setup & Imports

In [ ]:
!pip install -q scikit-survival

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sksurv.ensemble import (
    RandomSurvivalForest,
    GradientBoostingSurvivalAnalysis,
    ComponentwiseGradientBoostingSurvivalAnalysis
)
from sksurv.metrics import concordance_index_censored
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

HORIZONS = [12, 24, 48, 72]
N_FOLDS  = 5
SEEDS    = [42, 123, 456, 789, 2026]      # 5 seeds for final evaluation
SEEDS_FAST = [42, 123, 456]               # 3 seeds for Optuna search (faster)

print('All imports ready.')

## 2. Load Data

In [ ]:
train = pd.read_csv('/kaggle/input/WiDSWorldWide_GlobalDathon26/train.csv')
test  = pd.read_csv('/kaggle/input/WiDSWorldWide_GlobalDathon26/test.csv')
sample_sub = pd.read_csv('/kaggle/input/WiDSWorldWide_GlobalDathon26/sample_submission.csv')

print(f'Train shape : {train.shape}')
print(f'Test shape  : {test.shape}')
print(f'Event rate  : {train["event"].mean():.1%}  ({train["event"].sum()} hits / {len(train)} fires)')
print(f'Censored    : {(train["event"]==0).sum()} fires (fire did not reach evac zone within 72h)')
print(f'Features    : {test.shape[1] - 1} (same in train and test, minus targets)')

## 3. Exploratory Data Analysis

With only 221 samples, understanding the data deeply is crucial before building models.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('WiDS 2026 — Wildfire Dataset EDA', fontsize=16, fontweight='bold', y=1.02)

# 1. Survival time distribution
ax = axes[0, 0]
hits    = train[train['event'] == 1]['time_to_hit_hours']
censored = train[train['event'] == 0]['time_to_hit_hours']
ax.hist(hits,     bins=20, alpha=0.7, color='#e74c3c', label=f'Hit (n={len(hits)})')
ax.hist(censored, bins=20, alpha=0.5, color='#3498db', label=f'Censored (n={len(censored)})')
ax.axvline(x=72, color='black', linestyle='--', alpha=0.7, label='72h window')
ax.set_xlabel('Time to Hit (hours)')
ax.set_ylabel('Count')
ax.set_title('Survival Time Distribution')
ax.legend()

# 2. Distance to nearest evacuation zone
ax = axes[0, 1]
ax.hist(train[train['event']==1]['dist_min_ci_0_5h']/1000,
        bins=20, alpha=0.7, color='#e74c3c', label='Hit', density=True)
ax.hist(train[train['event']==0]['dist_min_ci_0_5h']/1000,
        bins=20, alpha=0.5, color='#3498db', label='Censored', density=True)
ax.set_xlabel('Min Distance to Evac Zone (km)')
ax.set_ylabel('Density')
ax.set_title('Distance to Evacuation Zone by Outcome')
ax.legend()

# 3. Closing speed distribution
ax = axes[0, 2]
ax.hist(train[train['event']==1]['closing_speed_m_per_h'],
        bins=25, alpha=0.7, color='#e74c3c', label='Hit', density=True)
ax.hist(train[train['event']==0]['closing_speed_m_per_h'],
        bins=25, alpha=0.5, color='#3498db', label='Censored', density=True)
ax.set_xlabel('Closing Speed (m/h)')
ax.set_title('Closing Speed by Outcome')
ax.legend()

# 4. Temporal resolution issue
ax = axes[1, 0]
low_res_counts = train['low_temporal_resolution_0_5h'].value_counts()
colors = ['#27ae60', '#e74c3c']
bars = ax.bar(['Has Dynamics\n(≥2 perimeters)', 'Low Resolution\n(1 perimeter)'],
              [low_res_counts.get(0, 0), low_res_counts.get(1, 0)],
              color=colors, alpha=0.8, edgecolor='white')
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            str(int(bar.get_height())), ha='center', fontweight='bold')
pct = low_res_counts.get(1, 0) / len(train) * 100
ax.set_title(f'Temporal Resolution\n({pct:.0f}% fires have only 1 perimeter!)')
ax.set_ylabel('Count')

# 5. Fire start by month (seasonality)
ax = axes[1, 1]
month_hits = train[train['event']==1]['event_start_month'].value_counts().sort_index()
month_cens = train[train['event']==0]['event_start_month'].value_counts().sort_index()
months = range(1, 13)
month_names = ['J','F','M','A','M','J','J','A','S','O','N','D']
hit_vals  = [month_hits.get(m, 0) for m in months]
cens_vals = [month_cens.get(m, 0) for m in months]
x = np.arange(12)
ax.bar(x - 0.2, hit_vals,  0.4, alpha=0.8, color='#e74c3c', label='Hit')
ax.bar(x + 0.2, cens_vals, 0.4, alpha=0.8, color='#3498db', label='Censored')
ax.set_xticks(x)
ax.set_xticklabels(month_names)
ax.set_title('Fire Starts by Month')
ax.set_ylabel('Count')
ax.legend()

# 6. Feature correlation with target
ax = axes[1, 2]
key_feats = ['dist_min_ci_0_5h', 'closing_speed_m_per_h', 'alignment_abs',
             'area_growth_rate_ha_per_h', 'dist_slope_ci_0_5h', 'dist_fit_r2_0_5h',
             'centroid_speed_m_per_h', 'projected_advance_m']
corrs = [train[f].corr(train['event']) for f in key_feats]
colors_corr = ['#e74c3c' if c < 0 else '#27ae60' for c in corrs]
short_names = ['dist_min', 'closing_spd', 'alignment', 'growth_rate',
               'dist_slope', 'fit_r2', 'centroid_spd', 'proj_advance']
ax.barh(short_names, corrs, color=colors_corr, alpha=0.8)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Correlation with Event')
ax.set_title('Feature-Target Correlation\n(raw features)')

plt.tight_layout()
plt.savefig('eda_overview.png', dpi=120, bbox_inches='tight')
plt.show()
print('Key observations:')
print(f'  • dist_min_ci_0_5h:      corr={train["dist_min_ci_0_5h"].corr(train["event"]):.3f} (closer = more dangerous)')
print(f'  • closing_speed_m_per_h: corr={train["closing_speed_m_per_h"].corr(train["event"]):.3f} (faster closing = more dangerous)')
print(f'  • alignment_abs:         corr={train["alignment_abs"].corr(train["event"]):.3f} (aligned = more dangerous)')
print(f'  • {pct:.0f}% of fires have only 1 perimeter (no dynamics features available!)')

## 4. Feature Engineering

We engineer 44 features from the 34 raw columns. The key insight: **distance is king**, but we need to transform it into various forms the model can use.

### Feature Groups

| Group | Features | Intuition |
|-------|----------|----------|
| Distance transforms | `log_dist_min`, `inv_dist`, proximity flags | Log-scale + inverse for nonlinear distance effects |
| Time-to-hit estimates | `est_time_to_hit`, `log_est_time` | Physics: dist / speed |
| Danger scores | `danger_score`, `threat_alignment`, `growth_danger` | Composite risk signals |
| Interaction terms | `speed_x_alignment`, `radial_x_inv_dist` | Capture joint effects |
| Temporal resolution | `has_dynamics`, `dynamics_close` | Flag: does the fire have meaningful spread data? |
| Seasonality | `hour_sin/cos`, `is_peak_season` | Circular encoding + fire season indicator |

### Why we keep features ≤ 44
With 221 samples, the **feature-to-sample ratio** must stay reasonable. We tested adding more features (physics-based, ratio features) — none helped because they were either correlated with existing features or caused overfitting.

In [ ]:
def engineer_features(df, extra_features=None):
    """
    Engineer 44 features from raw wildfire dynamics data.
    
    Parameters
    ----------
    df : pd.DataFrame  — raw train or test data
    extra_features : list[str] or None  — optional ablation features
    
    Returns
    -------
    pd.DataFrame with engineered features
    """
    d = df.copy()

    # ── Distance transforms ──────────────────────────────────────────────
    d['log_dist_min']  = np.log1p(d['dist_min_ci_0_5h'])
    d['already_close'] = (d['dist_min_ci_0_5h'] < 5000).astype(int)   # within 5km
    d['within_10km']   = (d['dist_min_ci_0_5h'] < 10000).astype(int)
    d['within_20km']   = (d['dist_min_ci_0_5h'] < 20000).astype(int)
    d['inv_dist']      = 1.0 / (d['dist_min_ci_0_5h'] + 100)          # avoid div-by-zero

    # ── Physics: estimated time-to-hit ───────────────────────────────────
    # Estimate: how long at current speed until the fire reaches the evac zone?
    d['est_time_to_hit'] = d['dist_min_ci_0_5h'] / np.maximum(d['closing_speed_m_per_h'], 1e-6)
    d['est_time_to_hit'] = np.clip(d['est_time_to_hit'], 0, 500)
    d.loc[d['closing_speed_m_per_h'] <= 0, 'est_time_to_hit'] = 500   # not closing
    d['log_est_time']    = np.log1p(d['est_time_to_hit'])

    # ── Composite danger scores ──────────────────────────────────────────
    d['danger_score']      = d['closing_speed_m_per_h'] / (d['log_dist_min'] + 1)
    d['threat_alignment']  = d['alignment_abs'] / (d['log_dist_min'] + 1)
    d['growth_danger']     = d['area_growth_rate_ha_per_h'] / (d['log_dist_min'] + 1)

    # ── Interaction features ─────────────────────────────────────────────
    d['speed_x_alignment'] = d['closing_speed_m_per_h'] * d['alignment_abs']
    d['radial_x_inv_dist'] = d['radial_growth_rate_m_per_h'] * d['inv_dist']

    # ── Temporal resolution features ─────────────────────────────────────
    # 72.8% of fires have only 1 perimeter → all dynamics features are 0!
    # We flag this and create interactions that degrade gracefully.
    d['has_dynamics']    = (d['low_temporal_resolution_0_5h'] == 0).astype(int)
    d['dynamics_close']  = d['has_dynamics'] / (d['log_dist_min'] + 1)
    d['n_perim_x_close'] = d['num_perimeters_0_5h'] * d['inv_dist']

    # ── Seasonality (circular encoding for hour, binary for season) ──────
    d['hour_sin']       = np.sin(2 * np.pi * d['event_start_hour'] / 24)
    d['hour_cos']       = np.cos(2 * np.pi * d['event_start_hour'] / 24)
    d['is_peak_season'] = d['event_start_month'].isin([6, 7, 8, 9, 10]).astype(int)

    # ── Optional ablation features ───────────────────────────────────────
    if extra_features and 'r2_x_alignment' in extra_features:
        d['r2_x_alignment'] = d['dist_fit_r2_0_5h'] * d['alignment_abs']
    if extra_features and 'dist_std_normalized' in extra_features:
        d['dist_std_normalized'] = d['dist_std_ci_0_5h'] / (d['dist_min_ci_0_5h'] + 100)
    if extra_features and 'centroid_to_radial_ratio' in extra_features:
        d['centroid_to_radial_ratio'] = np.clip(
            d['centroid_speed_m_per_h'] / (d['radial_growth_rate_m_per_h'] + 1e-6), 0, 50)
    if extra_features and 'is_afternoon' in extra_features:
        d['is_afternoon'] = ((d['event_start_hour'] >= 12) & (d['event_start_hour'] < 18)).astype(int)
    if extra_features and 'afternoon_x_danger' in extra_features:
        d['is_afternoon'] = ((d['event_start_hour'] >= 12) & (d['event_start_hour'] < 18)).astype(int)
        d['afternoon_x_danger'] = d['is_afternoon'] * d['danger_score']

    # ── Drop redundant / high-correlation raw columns ────────────────────
    drop_cols = [
        'relative_growth_0_5h',   # same as area_growth_rel_0_5h
        'area_first_ha',          # captured by log1p_area_first
        'area_growth_abs_0_5h',   # captured by log1p_growth
        'closing_speed_abs_m_per_h',  # redundant with closing_speed
        'dist_change_ci_0_5h',    # captured by dist_slope
        'spread_bearing_deg',     # encoded as sin/cos
        'event_start_hour',       # encoded as hour_sin/cos
        'event_start_dayofweek',  # low signal
    ]
    d.drop(columns=[c for c in drop_cols if c in d.columns], inplace=True)
    return d


def prepare_data(df_train, df_test, extra_features=None):
    train_fe = engineer_features(df_train, extra_features)
    test_fe  = engineer_features(df_test,  extra_features)
    TARGET_COLS = ['time_to_hit_hours', 'event']
    ID_COL      = 'event_id'
    feat_cols   = [c for c in train_fe.columns if c not in TARGET_COLS + [ID_COL]]
    X_tr    = train_fe[feat_cols].values
    y_ev    = train_fe['event'].values
    y_t     = train_fe['time_to_hit_hours'].values
    X_te    = test_fe[feat_cols].values
    ids     = test_fe[ID_COL].values
    y_surv  = np.array([(bool(e), t) for e, t in zip(y_ev, y_t)],
                       dtype=[('event', bool), ('time', float)])
    return X_tr, y_ev, y_t, y_surv, X_te, ids, feat_cols


X_train, y_event, y_time, y_surv, X_test, test_ids, FEAT_COLS = prepare_data(train, test)
print(f'Feature count : {len(FEAT_COLS)}')
print(f'Train matrix  : {X_train.shape}')
print(f'Test matrix   : {X_test.shape}')
print(f'\nFeatures:')
for i, f in enumerate(FEAT_COLS, 1):
    print(f'  {i:2d}. {f}')

## 5. Competition Metric & Training Framework

### The Metric

The competition metric is a hybrid of:
- **C-index** (30%): ranking quality — does a fire predicted as "more likely" actually hit sooner?
- **Weighted Brier Score** (70%): calibration quality — are the probabilities numerically accurate?

The Brier score at 48h has the highest weight (40%), making 48h predictions the most important.

### Multi-Seed Cross-Validation

With only 221 samples, a single 5-fold CV has high variance. We run **5 different random seeds × 5 folds = 25 fits per model** and average the OOF predictions. This dramatically reduces variance in our estimates.

In [ ]:
# ── Metric helpers ────────────────────────────────────────────────────────

def brier_censored(y_event, y_time, prob, horizon):
    """Brier score for right-censored survival data.
    
    We exclude samples that were censored *before* the horizon since
    we don't know their true outcome at that horizon.
    """
    mask  = ~((y_event == 0) & (y_time <= horizon))
    if mask.sum() == 0:
        return 0.0
    y_bin = ((y_event == 1) & (y_time <= horizon)).astype(float)
    return np.mean((prob[mask] - y_bin[mask]) ** 2)


def comp_metric(y_event, y_time, p12, p24, p48, p72):
    """Full competition metric: 0.3 * C-index + 0.7 * (1 - WBS)."""
    try:
        ci = concordance_index_censored(y_event.astype(bool), y_time, p72)[0]
    except Exception:
        ci = 0.5
    b24 = brier_censored(y_event, y_time, p24, 24)
    b48 = brier_censored(y_event, y_time, p48, 48)
    b72 = brier_censored(y_event, y_time, p72, 72)
    wb  = 0.3 * b24 + 0.4 * b48 + 0.3 * b72
    return 0.3 * ci + 0.7 * (1 - wb), ci, wb, b24, b48, b72


def enforce_monotonicity(p12, p24, p48, p72):
    """Enforce prob_12h ≤ prob_24h ≤ prob_48h ≤ prob_72h."""
    p24 = np.maximum(p24, p12)
    p48 = np.maximum(p48, p24)
    p72 = np.maximum(p72, p48)
    return p12, p24, p48, p72


def extract_survival_probs(surv_funcs):
    """Convert survival functions S(t) → hit probabilities P(T ≤ t)."""
    probs = {h: [] for h in HORIZONS}
    for fn in surv_funcs:
        for h in HORIZONS:
            try:
                p = 1.0 - fn(h)
            except Exception:
                p = 1.0 - fn.y[-1] if hasattr(fn, 'y') else 0.5
            probs[h].append(np.clip(p, 0.001, 0.999))
    return {h: np.array(v) for h, v in probs.items()}


# ── Multi-seed training for sksurv models ────────────────────────────────

def train_survival_multiseed(model_class, model_params, name,
                              X, y_surv, y_ev, y_t, X_te,
                              seeds=SEEDS, needs_scaling=False, verbose=True):
    """
    Train a scikit-survival model with multi-seed 5-fold CV.
    
    Returns OOF and test predictions averaged over all seeds × folds.
    """
    n, n_te = len(X), len(X_te)
    all_oof  = {h: np.zeros(n)    for h in HORIZONS}
    all_test = {h: np.zeros(n_te) for h in HORIZONS}

    for seed in seeds:
        skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
        seed_oof  = {h: np.zeros(n)    for h in HORIZONS}
        seed_test = {h: np.zeros(n_te) for h in HORIZONS}

        for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y_ev)):
            if needs_scaling:
                sc = StandardScaler()
                X_tr      = sc.fit_transform(X[tr_idx])
                X_va      = sc.transform(X[va_idx])
                X_te_fold = sc.transform(X_te)
            else:
                X_tr, X_va = X[tr_idx], X[va_idx]
                X_te_fold  = X_te

            params = {**model_params, 'random_state': seed}
            model  = model_class(**params)
            model.fit(X_tr, y_surv[tr_idx])

            va_probs = extract_survival_probs(model.predict_survival_function(X_va))
            te_probs = extract_survival_probs(model.predict_survival_function(X_te_fold))
            for h in HORIZONS:
                seed_oof[h][va_idx] = va_probs[h]
                seed_test[h]       += te_probs[h] / N_FOLDS

        for h in HORIZONS:
            all_oof[h]  += seed_oof[h]  / len(seeds)
            all_test[h] += seed_test[h] / len(seeds)

    all_oof[12],  all_oof[24],  all_oof[48],  all_oof[72]  = enforce_monotonicity(*[all_oof[h]  for h in HORIZONS])
    all_test[12], all_test[24], all_test[48], all_test[72] = enforce_monotonicity(*[all_test[h] for h in HORIZONS])

    score = comp_metric(y_ev, y_t, all_oof[12], all_oof[24], all_oof[48], all_oof[72])
    if verbose:
        print(f'  {name:45s}  Hybrid={score[0]:.4f}  C-index={score[1]:.4f}  Brier={score[2]:.4f}')
    return {'oof': all_oof, 'test': all_test, 'score': score}


# ── Multi-horizon LGBM ───────────────────────────────────────────────────

def train_lgbm_multiseed(params, X, y_ev, y_t, X_te, seeds=SEEDS, verbose=True):
    """
    Train separate LightGBM binary classifiers for each horizon (12h, 24h, 48h).
    72h is extrapolated from 48h since the dataset has very few 48-72h events.
    
    Note: We exclude censored-before-horizon samples from each horizon's training set.
    """
    n, n_te = len(X), len(X_te)
    all_oof  = {h: np.zeros(n)    for h in HORIZONS}
    all_test = {h: np.zeros(n_te) for h in HORIZONS}

    # Estimate 72h from 48h: growth factor from observed hit rate in (48h, 72h]
    hits_by_48 = ((y_ev == 1) & (y_t <= 48)).sum()
    hits_48_72 = ((y_ev == 1) & (y_t > 48) & (y_t <= 72)).sum()
    at_risk_48 = n - hits_by_48
    gf = hits_48_72 / max(at_risk_48, 1)

    for seed in seeds:
        skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
        for h in [12, 24, 48]:
            y_bin = ((y_ev == 1) & (y_t <= h)).astype(int)
            valid = ~((y_ev == 0) & (y_t <= h))  # exclude ambiguous censored
            for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y_ev)):
                tr_v = valid[tr_idx]; va_v = valid[va_idx]
                X_tr, y_tr = X[tr_idx][tr_v], y_bin[tr_idx][tr_v]
                X_va, y_va = X[va_idx][va_v], y_bin[va_idx][va_v]
                m = lgb.LGBMClassifier(
                    objective='binary', metric='binary_logloss',
                    verbose=-1, n_estimators=500, **params, random_state=seed
                )
                m.fit(X_tr, y_tr, eval_set=[(X_va, y_va)],
                      callbacks=[lgb.early_stopping(50, verbose=False)])
                all_oof[h][va_idx]  += m.predict_proba(X[va_idx])[:,1] / len(seeds)
                all_test[h]         += m.predict_proba(X_te)[:,1] / (N_FOLDS * len(seeds))

    # 72h extrapolation
    all_oof[72]  = all_oof[48]  + (1 - all_oof[48])  * gf
    all_test[72] = all_test[48] + (1 - all_test[48]) * gf
    all_oof[12],  all_oof[24],  all_oof[48],  all_oof[72]  = enforce_monotonicity(*[all_oof[h]  for h in HORIZONS])
    all_test[12], all_test[24], all_test[48], all_test[72] = enforce_monotonicity(*[all_test[h] for h in HORIZONS])

    score = comp_metric(y_ev, y_t, all_oof[12], all_oof[24], all_oof[48], all_oof[72])
    if verbose:
        print(f'  {"LightGBM multi-horizon":45s}  Hybrid={score[0]:.4f}  C-index={score[1]:.4f}  Brier={score[2]:.4f}')
    return {'oof': all_oof, 'test': all_test, 'score': score}


print('Metric helpers and training framework ready.')

## 6. Adversarial Validation

**Question**: Is our OOF score a reliable proxy for the leaderboard score? Or is there a distribution shift between train and test?

**Method**: Train a binary classifier to distinguish train from test samples. If AUC ≈ 0.5, the distributions are indistinguishable — no shift. If AUC > 0.65, there's meaningful shift.

This is important because if there's distribution shift, we might need to:
- Down-weight train samples that look unlike test
- Remove features causing the shift
- Use adversarial sample weights

In [ ]:
from sklearn.metrics import roc_auc_score

# Label: 0 = train, 1 = test
X_adv = np.vstack([X_train, X_test])
y_adv = np.array([0] * len(X_train) + [1] * len(X_test))

adv_aucs  = []
feat_imps = np.zeros(X_adv.shape[1])

skf_adv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for fold, (tr_idx, va_idx) in enumerate(skf_adv.split(X_adv, y_adv)):
    m = lgb.LGBMClassifier(
        n_estimators=200, num_leaves=15, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, random_state=42, verbose=-1
    )
    m.fit(X_adv[tr_idx], y_adv[tr_idx])
    preds = m.predict_proba(X_adv[va_idx])[:, 1]
    adv_aucs.append(roc_auc_score(y_adv[va_idx], preds))
    feat_imps += m.feature_importances_ / 5

mean_auc = np.mean(adv_aucs)
print(f'Adversarial Validation AUC: {mean_auc:.4f} ± {np.std(adv_aucs):.4f}')

if mean_auc < 0.55:
    print('  ✅ No meaningful distribution shift — train and test look similar.')
    print('  The OOF-LB gap is likely just noise from the small test set (95 rows).')
elif mean_auc < 0.65:
    print('  ⚠️  Moderate distribution shift detected.')
else:
    print('  ❌ High distribution shift — consider adversarial weighting!')

# Plot feature importances for the adversarial classifier
feat_imp_df = pd.DataFrame({'feature': FEAT_COLS, 'importance': feat_imps})
feat_imp_df = feat_imp_df.sort_values('importance', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: bar chart of top features
top10 = feat_imp_df.head(10)
axes[0].barh(top10['feature'][::-1], top10['importance'][::-1],
             color='#e74c3c', alpha=0.8)
axes[0].set_title(f'Top-10 Features: Train vs Test Distinguishability\n(AUC={mean_auc:.4f} — lower is better)',
                  fontweight='bold')
axes[0].set_xlabel('Feature Importance in Adversarial Classifier')

# Right: distribution comparison for most "shifted" feature
top_feat = feat_imp_df.iloc[0]['feature']
idx = FEAT_COLS.index(top_feat)
axes[1].hist(X_train[:, idx], bins=30, alpha=0.6, density=True,
             color='#3498db', label='Train')
axes[1].hist(X_test[:, idx],  bins=30, alpha=0.6, density=True,
             color='#e74c3c', label='Test')
axes[1].set_title(f'Distribution of "{top_feat}"\n(most important in adversarial model)',
                  fontweight='bold')
axes[1].set_xlabel(top_feat)
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'\nConclusion: AUC={mean_auc:.4f} → no corrective action needed.')

## 7. Feature Ablation Study

Before adding new features, we rigorously test each candidate in isolation.

### Candidates Tested

| Feature | Intuition | Correlation check |
|---------|-----------|------------------|
| `r2_x_alignment` | Trajectory confidence × direction — is the fire on a confident aligned approach? | Moderate ✅ |
| `dist_std_normalized` | Spread of evacuation zones relative to nearest — zone density | HIGH ⚠️ (corr=0.92 with `danger_score`) |
| `centroid_to_radial_ratio` | Is the fire translating (moving) or just growing in place? | Moderate ✅ |
| `is_afternoon` | Peak fire activity window (12-18h) | Low ✅ |
| `afternoon_x_danger` | Afternoon × danger interaction | Low ✅ |

### Why we use ablation instead of forward selection
With 221 samples and ~44 features, adding even one useless feature can hurt due to overfitting. We use **3-seed LGBM as a fast proxy** to test each feature individually before committing to the full 5-seed evaluation.

In [ ]:
# Check correlations of new features with existing important features
train_with_extras = engineer_features(train, extra_features=[
    'r2_x_alignment', 'dist_std_normalized', 'centroid_to_radial_ratio',
    'is_afternoon', 'afternoon_x_danger'
])
train_vals = train_with_extras.drop(columns=['event_id', 'time_to_hit_hours', 'event'], errors='ignore')

new_feats = ['r2_x_alignment', 'dist_std_normalized', 'centroid_to_radial_ratio',
             'is_afternoon', 'afternoon_x_danger']
top_v2 = ['log_dist_min', 'already_close', 'est_time_to_hit',
           'danger_score', 'threat_alignment']

print('Correlation of new features with top-5 existing features:')
print(f'  {"New Feature":35s}  {"max_corr":10s}  {"with_event":10s}  Status')
print(f'  {"-"*75}')
for feat in new_feats:
    if feat not in train_vals.columns:
        continue
    max_corr = max(abs(train_vals[feat].corr(train_vals[f]))
                  for f in top_v2 if f in train_vals.columns)
    event_corr = train_vals[feat].corr(train_with_extras['event'])
    flag = '⚠️  HIGH' if max_corr > 0.7 else '✅ OK'
    print(f'  {feat:35s}  {max_corr:.3f}       {event_corr:+.3f}       {flag}')

In [ ]:
# Optuna-tuned LGBM params (found in prior experiments)
LGBM_PARAMS = {
    'learning_rate':     0.0617,
    'num_leaves':        15,
    'max_depth':         4,
    'min_child_samples': 13,
    'subsample':         0.768,
    'colsample_bytree':  0.580,
    'reg_alpha':         0.389,
    'reg_lambda':        1.055,
}

def quick_lgbm_oof(extra_features=None, seeds=SEEDS_FAST, label=''):
    """Run 3-seed LGBM CV — quick proxy for feature ablation."""
    X_tr, y_ev, y_t, _, X_te, _, _ = prepare_data(train, test, extra_features)
    res   = train_lgbm_multiseed(LGBM_PARAMS, X_tr, y_ev, y_t, X_te,
                                  seeds=seeds, verbose=False)
    score = res['score'][0]
    print(f'  {label:42s}  n_feats={X_tr.shape[1]}  OOF={score:.4f}')
    return score, res

print('Ablation study — 3-seed LGBM fast proxy:')
print(f'  {"Experiment":42s}  {"n_feats":8s}  OOF')
print(f'  {"-"*62}')

base_score, base_res = quick_lgbm_oof(None,                          label='Baseline (44 features)')
r2_score,   r2_res   = quick_lgbm_oof(['r2_x_alignment'],            label='+ r2_x_alignment')
std_score,  std_res  = quick_lgbm_oof(['dist_std_normalized'],       label='+ dist_std_normalized')
ctr_score,  ctr_res  = quick_lgbm_oof(['centroid_to_radial_ratio'],  label='+ centroid_to_radial_ratio')
aft_score,  aft_res  = quick_lgbm_oof(['is_afternoon'],              label='+ is_afternoon')
axd_score,  axd_res  = quick_lgbm_oof(['afternoon_x_danger'],        label='+ afternoon_x_danger')

improving = [f for f, s in [
    ('r2_x_alignment', r2_score), ('dist_std_normalized', std_score),
    ('centroid_to_radial_ratio', ctr_score), ('is_afternoon', aft_score),
    ('afternoon_x_danger', axd_score)
] if s > base_score]

print(f'\nFeatures that improved over baseline: {improving if improving else "none"}')
print('\nInsight: With only 221 samples, our 44 features are already optimal.')
print('Adding more features increases the feature-to-sample ratio and causes overfitting.')

## 8. Optuna Hyperparameter Tuning

We use [Optuna](https://optuna.org/) to automatically find optimal hyperparameters for three survival models.

### Strategy
- **Search phase**: 3 seeds × 5 folds (15 fits per trial) for speed
- **Evaluation phase**: 5 seeds × 5 folds (25 fits) for accurate final scores
- **Sampler**: Default TPE (Tree-structured Parzen Estimator) — Bayesian optimization

### Models
1. **ComponentwiseGBM (CompGBM)**: Gradient boosting that fits one feature at a time — excellent for small datasets, implicit feature selection
2. **GradientBoostingSurvivalAnalysis (GBSA)**: Full gradient boosting with survival loss — more expressive than CompGBM
3. **RandomSurvivalForest (RSF)**: Ensemble of survival trees — robust, non-parametric, handles interactions well

In [ ]:
# ── Optuna: ComponentwiseGBM ──────────────────────────────────────────────

def objective_compgbm(trial):
    params = {
        'n_estimators':  trial.suggest_int('n_estimators', 100, 600),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample':     trial.suggest_float('subsample', 0.5, 1.0),
        'random_state':  42
    }
    scores = []
    for seed in SEEDS_FAST:
        skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
        oof = {h: np.zeros(len(X_train)) for h in HORIZONS}
        for fold, (tr_idx, va_idx) in enumerate(skf.split(X_train, y_event)):
            sc   = StandardScaler()
            X_tr = sc.fit_transform(X_train[tr_idx])
            X_va = sc.transform(X_train[va_idx])
            m    = ComponentwiseGradientBoostingSurvivalAnalysis(**{**params, 'random_state': seed})
            m.fit(X_tr, y_surv[tr_idx])
            va_probs = extract_survival_probs(m.predict_survival_function(X_va))
            for h in HORIZONS:
                oof[h][va_idx] = va_probs[h]
        oof[12], oof[24], oof[48], oof[72] = enforce_monotonicity(*[oof[h] for h in HORIZONS])
        scores.append(comp_metric(y_event, y_time, oof[12], oof[24], oof[48], oof[72])[0])
    return np.mean(scores)


print('Tuning ComponentwiseGBM (60 trials, 3-seed CV)...')
study_compgbm = optuna.create_study(direction='maximize')
study_compgbm.optimize(objective_compgbm, n_trials=60, show_progress_bar=False)
COMPGBM_BEST = {**study_compgbm.best_params, 'random_state': 42}
print(f'  Best OOF (3-seed): {study_compgbm.best_value:.4f}')
print(f'  Best params: n_estimators={COMPGBM_BEST["n_estimators"]}, '
      f'lr={COMPGBM_BEST["learning_rate"]:.4f}, subsample={COMPGBM_BEST["subsample"]:.3f}')

In [ ]:
# ── Optuna: GradientBoostingSurvivalAnalysis ──────────────────────────────

def objective_gbsa(trial):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 50, 300),
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'max_depth':         trial.suggest_int('max_depth', 2, 4),
        'min_samples_split': trial.suggest_int('min_samples_split', 8, 30),
        'min_samples_leaf':  trial.suggest_int('min_samples_leaf', 4, 20),
        'subsample':         trial.suggest_float('subsample', 0.5, 0.9),
        'max_features':      trial.suggest_categorical('max_features', ['sqrt', 'log2']),
        'random_state':      42
    }
    scores = []
    for seed in SEEDS_FAST:
        skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
        oof = {h: np.zeros(len(X_train)) for h in HORIZONS}
        for fold, (tr_idx, va_idx) in enumerate(skf.split(X_train, y_event)):
            m = GradientBoostingSurvivalAnalysis(**{**params, 'random_state': seed})
            m.fit(X_train[tr_idx], y_surv[tr_idx])
            va_probs = extract_survival_probs(m.predict_survival_function(X_train[va_idx]))
            for h in HORIZONS:
                oof[h][va_idx] = va_probs[h]
        oof[12], oof[24], oof[48], oof[72] = enforce_monotonicity(*[oof[h] for h in HORIZONS])
        scores.append(comp_metric(y_event, y_time, oof[12], oof[24], oof[48], oof[72])[0])
    return np.mean(scores)


print('Tuning GradientBoostingSurvivalAnalysis (80 trials, 3-seed CV)...')
study_gbsa = optuna.create_study(direction='maximize')
study_gbsa.optimize(objective_gbsa, n_trials=80, show_progress_bar=False)
GBSA_BEST = {**study_gbsa.best_params, 'random_state': 42}
print(f'  Best OOF (3-seed): {study_gbsa.best_value:.4f}')
print(f'  Best params: n_estimators={GBSA_BEST["n_estimators"]}, '
      f'lr={GBSA_BEST["learning_rate"]:.4f}, max_depth={GBSA_BEST["max_depth"]}')

In [ ]:
# ── Optuna: RandomSurvivalForest ──────────────────────────────────────────

def objective_rsf(trial):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 200, 800),
        'min_samples_split': trial.suggest_int('min_samples_split', 5, 20),
        'min_samples_leaf':  trial.suggest_int('min_samples_leaf', 2, 10),
        'max_depth':         trial.suggest_int('max_depth', 4, 8),
        'max_features':      trial.suggest_categorical('max_features', ['sqrt', 'log2']),
        'random_state':      42,
        'n_jobs':            -1
    }
    scores = []
    for seed in [42, 123]:  # Only 2 seeds — RSF is slow
        skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
        oof = {h: np.zeros(len(X_train)) for h in HORIZONS}
        for fold, (tr_idx, va_idx) in enumerate(skf.split(X_train, y_event)):
            m = RandomSurvivalForest(**{**params, 'random_state': seed})
            m.fit(X_train[tr_idx], y_surv[tr_idx])
            va_probs = extract_survival_probs(m.predict_survival_function(X_train[va_idx]))
            for h in HORIZONS:
                oof[h][va_idx] = va_probs[h]
        oof[12], oof[24], oof[48], oof[72] = enforce_monotonicity(*[oof[h] for h in HORIZONS])
        scores.append(comp_metric(y_event, y_time, oof[12], oof[24], oof[48], oof[72])[0])
    return np.mean(scores)


print('Tuning RandomSurvivalForest (40 trials, 2-seed CV)...')
study_rsf = optuna.create_study(direction='maximize')
study_rsf.optimize(objective_rsf, n_trials=40, show_progress_bar=False)
RSF_BEST = {**study_rsf.best_params, 'random_state': 42, 'n_jobs': -1}
print(f'  Best OOF (2-seed): {study_rsf.best_value:.4f}')
print(f'  Best params: n_estimators={RSF_BEST["n_estimators"]}, '
      f'max_depth={RSF_BEST["max_depth"]}, max_features={RSF_BEST["max_features"]}')

In [ ]:
# Visualize Optuna optimization history
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Optuna Hyperparameter Optimization History', fontsize=14, fontweight='bold')

for ax, study, name in zip(axes,
                            [study_compgbm, study_gbsa, study_rsf],
                            ['ComponentwiseGBM', 'GBSA', 'RSF']):
    trials = [t.value for t in study.trials if t.value is not None]
    best_so_far = np.maximum.accumulate(trials)
    ax.scatter(range(len(trials)), trials, alpha=0.4, s=20, color='#3498db', label='Trial')
    ax.plot(range(len(trials)), best_so_far, color='#e74c3c', linewidth=2, label='Best so far')
    ax.set_title(f'{name}\nBest: {max(trials):.4f}')
    ax.set_xlabel('Trial')
    ax.set_ylabel('OOF Score (3-seed)')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## 9. Full Model Training (5-Seed CV)

Now we train all models with the tuned parameters using the full 5-seed CV for accurate evaluation.

In [ ]:
print('Training all models (5-seed, 5-fold CV each)...')
print('='*70)

models = {}

# LightGBM (Optuna-tuned from prior experiments)
models['LGBM'] = train_lgbm_multiseed(LGBM_PARAMS, X_train, y_event, y_time, X_test)

# CompGBM with Optuna-tuned params
models['CompGBM'] = train_survival_multiseed(
    ComponentwiseGradientBoostingSurvivalAnalysis, COMPGBM_BEST,
    'ComponentwiseGBM (Optuna)', X_train, y_surv, y_event, y_time, X_test,
    needs_scaling=True
)

# GBSA with Optuna-tuned params
models['GBSA'] = train_survival_multiseed(
    GradientBoostingSurvivalAnalysis, GBSA_BEST,
    'GBSA (Optuna)', X_train, y_surv, y_event, y_time, X_test
)

# RSF with Optuna-tuned params
models['RSF'] = train_survival_multiseed(
    RandomSurvivalForest, RSF_BEST,
    'RSF (Optuna)', X_train, y_surv, y_event, y_time, X_test
)

# RSF_deep — slightly deeper variant for diversity
rsf_deep = {**RSF_BEST,
            'max_depth': min(RSF_BEST.get('max_depth', 6) + 1, 10),
            'min_samples_leaf': max(RSF_BEST.get('min_samples_leaf', 2) - 1, 1)}
models['RSF_deep'] = train_survival_multiseed(
    RandomSurvivalForest, rsf_deep,
    'RSF_deep (Optuna-based)', X_train, y_surv, y_event, y_time, X_test
)

print('='*70)
print('\nModel ranking (5-seed OOF):')
for name, m in sorted(models.items(), key=lambda x: -x[1]['score'][0]):
    s = m['score']
    print(f'  {name:20s}  Hybrid={s[0]:.4f}  C-index={s[1]:.4f}  Brier={s[2]:.4f}')

## 10. Diversity-Aware Ensemble

### The Ensemble Diversity Problem

A critical insight in machine learning ensembling: **ensemble benefit comes from model diversity**.

When we tune all models with Optuna to maximize the same objective:
- ✅ Each individual model improves
- ❌ All models start making similar predictions
- ❌ Correlated errors → ensemble benefit shrinks

This is why sometimes a model with "worse" individual score adds more value to the ensemble than a model with a "better" individual score — it makes *different* mistakes.

> **Rule of thumb**: In a small-data survival ensemble, prefer a mix of architecturally diverse models over all-tuned models.

In [ ]:
# ── Top-5 simple average ensemble ────────────────────────────────────────
top5_names = sorted(models.keys(), key=lambda m: -models[m]['score'][0])[:5]
print(f'Top-5 models selected: {top5_names}')

ens_oof  = {h: np.mean([models[m]['oof'][h]  for m in top5_names], axis=0) for h in HORIZONS}
ens_test = {h: np.mean([models[m]['test'][h] for m in top5_names], axis=0) for h in HORIZONS}

ens_oof[12],  ens_oof[24],  ens_oof[48],  ens_oof[72]  = enforce_monotonicity(*[ens_oof[h]  for h in HORIZONS])
ens_test[12], ens_test[24], ens_test[48], ens_test[72] = enforce_monotonicity(*[ens_test[h] for h in HORIZONS])

ens_score = comp_metric(y_event, y_time, ens_oof[12], ens_oof[24], ens_oof[48], ens_oof[72])
print(f'\nEnsemble OOF: Hybrid={ens_score[0]:.4f}  C-index={ens_score[1]:.4f}  Brier={ens_score[2]:.4f}')

# Visualize: individual vs ensemble
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: model comparison
all_model_names  = list(models.keys()) + ['Ensemble']
all_model_scores = [models[m]['score'][0] for m in models.keys()] + [ens_score[0]]
colors_bar = ['#3498db'] * len(models) + ['#e74c3c']

bars = axes[0].bar(all_model_names, all_model_scores, color=colors_bar, alpha=0.85)
axes[0].set_ylim(min(all_model_scores) * 0.995, max(all_model_scores) * 1.005)
for bar, score in zip(bars, all_model_scores):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.0001,
                 f'{score:.4f}', ha='center', va='bottom', fontsize=8, fontweight='bold')
axes[0].set_title('OOF Scores: Individual Models vs Ensemble', fontweight='bold')
axes[0].set_ylabel('Hybrid Score (OOF)')
axes[0].tick_params(axis='x', rotation=15)

# Right: prediction correlation heatmap
pred_matrix = np.column_stack([models[m]['oof'][48] for m in models.keys()])
corr_matrix = np.corrcoef(pred_matrix.T)
im = axes[1].imshow(corr_matrix, cmap='RdYlGn_r', vmin=0.7, vmax=1.0)
axes[1].set_xticks(range(len(models)))
axes[1].set_yticks(range(len(models)))
axes[1].set_xticklabels(list(models.keys()), rotation=45)
axes[1].set_yticklabels(list(models.keys()))
for i in range(len(models)):
    for j in range(len(models)):
        axes[1].text(j, i, f'{corr_matrix[i,j]:.2f}', ha='center', va='center',
                     fontsize=9, color='black')
plt.colorbar(im, ax=axes[1])
axes[1].set_title('OOF Prediction Correlation (48h)\nLower = more diverse = better ensemble', fontweight='bold')

plt.tight_layout()
plt.show()

## 11. Generate Submission

Final checks before saving:
1. Clip probabilities to [0.001, 0.999] (avoid log(0) issues)
2. Verify monotonicity: `prob_12h ≤ prob_24h ≤ prob_48h ≤ prob_72h`
3. Verify row count and column names match sample submission

In [ ]:
# Clip and verify
for h in HORIZONS:
    ens_test[h] = np.clip(ens_test[h], 0.001, 0.999)

violations = sum(
    1 for i in range(len(X_test))
    if not all(ens_test[HORIZONS[j]][i] <= ens_test[HORIZONS[j+1]][i] + 1e-10
               for j in range(3))
)
print(f'Monotonicity violations: {violations}/{len(X_test)}  (must be 0)')

submission = pd.DataFrame({
    'event_id': test_ids,
    'prob_12h': ens_test[12],
    'prob_24h': ens_test[24],
    'prob_48h': ens_test[48],
    'prob_72h': ens_test[72],
})

assert list(submission.columns) == list(sample_sub.columns), 'Column mismatch!'
assert len(submission) == len(sample_sub), f'Row count mismatch: {len(submission)} vs {len(sample_sub)}'

submission.to_csv('../submission_v4_all_optuna.csv', index=False)
print(f'Saved: submission_v4_all_optuna.csv  ({len(submission)} rows)')

print('\nSample predictions:')
print(submission.head(8).to_string(index=False))

# Distribution check
print(f'\nProbability statistics:')
for h in HORIZONS:
    col = f'prob_{h}h'
    print(f'  {col}: mean={submission[col].mean():.3f}  '
          f'min={submission[col].min():.3f}  max={submission[col].max():.3f}')

In [ ]:
# Visualize the submission predictions
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Submission Prediction Analysis', fontsize=14, fontweight='bold')

# Left: probability distributions by horizon
colors_h = ['#2ecc71', '#f39c12', '#e67e22', '#e74c3c']
for h, color in zip(HORIZONS, colors_h):
    axes[0].hist(submission[f'prob_{h}h'], bins=25, alpha=0.5,
                 label=f'prob_{h}h', color=color, density=True)
axes[0].set_xlabel('Predicted Probability')
axes[0].set_ylabel('Density')
axes[0].set_title('Probability Distributions by Horizon')
axes[0].legend()

# Right: monotonicity check — prob_72h vs prob_12h
axes[1].scatter(submission['prob_12h'], submission['prob_72h'],
                alpha=0.7, s=30, color='#9b59b6')
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.4, label='prob_12h = prob_72h')
axes[1].set_xlabel('prob_12h')
axes[1].set_ylabel('prob_72h')
axes[1].set_title('Monotonicity Check\n(all points should be above diagonal)')
axes[1].legend()

plt.tight_layout()
plt.show()

## 12. Results & Key Takeaways

---

### 🏆 Results Summary

| Model | OOF Score | Notes |
|-------|-----------|-------|
| LightGBM (multi-horizon) | 0.9716 | Optuna-tuned, colsample_bytree=0.58 is key |
| ComponentwiseGBM | 0.9719 | Optuna: n_estimators=464, lr=0.091 |
| GBSA | 0.9712 | Optuna: n_estimators=284, max_depth=2 |
| RSF | 0.9674 | Optuna: n_estimators=425, max_features=log2 |
| **Ensemble (Top-5 avg)** | **0.9723** | Simple average of top-5 models |

> **Public Leaderboard: 96.35%** ✅

---

### 💡 Key Lessons Learned

#### 1. Distance is the dominant signal
With 221 samples, the most important feature is `dist_min_ci_0_5h`. Log-transforming it and creating proximity flags captures its non-linear effect.

#### 2. Overfitting is the #1 enemy
72.8% of fires have only 1 perimeter observation — all dynamics features are zero for these fires. Adding new features beyond the 44 we have consistently hurt performance.

#### 3. Multi-seed averaging reduces variance
With 221 samples, a single 5-fold split is noisy. Using 5 seeds × 5 folds = 25 fits per model dramatically stabilizes OOF estimates.

#### 4. Adversarial validation diagnoses your OOF-LB gap
Our adversarial AUC = 0.475 (≈ random) confirmed there's no train/test distribution shift. The ~1% OOF-LB gap is just noise from the small test set (95 rows).

#### 5. Feature ablation before adding (not after)
Every feature candidate was tested individually with a fast 3-seed proxy before committing to the full evaluation. This saves compute and prevents data leakage in feature selection.

#### 6. Simple average beats complex ensemble optimization
Nelder-Mead, per-horizon weight optimization — all underperformed simple averaging. With 221 samples, there's not enough data to reliably estimate ensemble weights.

#### 7. The Ensemble Diversity Paradox
Tuning all models with Optuna improved every individual model — but the ensemble score *dropped*. Why? Optuna drives models toward the same objective, correlating their errors. The fix: keep some models at default parameters as "diversity boosters."

---

### 📦 What's in This Notebook
- Full EDA with visualizations
- 44-feature engineering pipeline
- Adversarial validation framework
- Rigorous feature ablation protocol
- Optuna tuning for 3 survival models
- Diversity-aware ensembling discussion
- Clean, reproducible submission pipeline

---

*If this notebook was helpful, an upvote is appreciated!* 🙏